# SentinelVision — Phase 2: fine-tune YOLOv8 on exam-proctoring data

Trains a 6-class detector (`book`, `cell phone`, `headphone`, `laptop`, `person`, `tv`) on the
[Online Proctoring System](https://universe.roboflow.com/online-exam-cheating-detection-kvdul/online-proctoring-system-x27ou-e7abr)
dataset (CC BY 4.0, ~25k frames).

**Why this runs here and not locally:** the dev laptop has Intel integrated graphics and a
CPU-only PyTorch build. On a Kaggle T4 this is a couple of hours.

**Why the split is rebuilt:** the export's own train/valid/test are leaky — one source video
(`final-2_mp4`) is ~87% of the data *and* supplies 100% of valid and test. Validating on frames
from the same video of the same person in the same room reports a number that says nothing about
anyone else. Cell 4 regroups by source: the dominant video trains, the other sources validate.
So the val score answers *"trained on one person, does it work on someone new?"*

**Baseline to beat:** pre-trained YOLOv8n scores **F1 0.193** for `cell phone` on this domain
(precision 0.215, recall 0.176 at its best threshold) — measured locally, see `docs/DATASETS.md`.

---
### Before you run
1. Settings → **Accelerator → GPU T4 x2** (or P100)
2. Add-ons → **Secrets** → add `ROBOFLOW_API_KEY` with your key
3. Settings → **Internet → On** (needed to download the dataset)

In [ ]:
# 1. Environment
!pip install -q ultralytics

import torch, os
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('No GPU. Settings -> Accelerator -> GPU before running.')

In [ ]:
# 2. API key from Kaggle Secrets -- never paste it into a cell, notebooks get shared
from kaggle_secrets import UserSecretsClient

API_KEY = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
print('key loaded:', bool(API_KEY))

In [ ]:
# 3. Download the dataset (~1.4 GB)
import json, shutil, urllib.parse, urllib.request, zipfile, os

WORKSPACE = 'online-exam-cheating-detection-kvdul'
PROJECT   = 'online-proctoring-system-x27ou-e7abr'
VERSION   = 1
DEST      = '/kaggle/working/proctoring'

url = f'https://api.roboflow.com/{WORKSPACE}/{PROJECT}/{VERSION}/yolov8'
req = f"{url}?{urllib.parse.urlencode({'api_key': API_KEY})}"

with urllib.request.urlopen(req, timeout=180) as r:
    link = json.load(r)['export']['link']

os.makedirs(DEST, exist_ok=True)
archive = '/kaggle/working/export.zip'
with urllib.request.urlopen(link, timeout=600) as r, open(archive, 'wb') as f:
    shutil.copyfileobj(r, f)
print(f'{os.path.getsize(archive)/1e6:.0f} MB downloaded')

with zipfile.ZipFile(archive) as z:
    z.extractall(DEST)
os.remove(archive)
print(sorted(os.listdir(DEST)))

In [ ]:
# 4. Rebuild the split so train and val share no source video.
#    Mirrors src/detection/split_by_source.py in the repo.
import re, yaml

SOURCE_PATTERN = re.compile(r'^(.*?)[-_]?(\d+)_jpg\.rf\.')

def source_of(name):
    m = SOURCE_PATTERN.match(name)
    return m.group(1) if (m and m.group(1)) else '_misc'

by_source = {}
for split in ('train', 'valid', 'test'):
    d = os.path.join(DEST, split, 'images')
    if not os.path.isdir(d):
        continue
    for name in os.listdir(d):
        if name.lower().endswith(('.jpg', '.jpeg', '.png')):
            by_source.setdefault(source_of(name), []).append(os.path.join(d, name))

largest   = max(by_source, key=lambda s: len(by_source[s]))
train_p   = by_source[largest]
val_p     = [p for s, ps in by_source.items() if s != largest for p in ps]

print(f"{'source':<36}{'images':>8}  split")
print('-' * 54)
for s, ps in sorted(by_source.items(), key=lambda kv: -len(kv[1])):
    print(f"{s or '(blank)':<36}{len(ps):>8}  {'TRAIN' if s == largest else 'val'}")
print(f'\ntrain {len(train_p)} | val {len(val_p)} from {len(by_source)-1} sources')

names = yaml.safe_load(open(os.path.join(DEST, 'data.yaml')))['names']
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]

open('/kaggle/working/train.txt', 'w').write('\n'.join(train_p) + '\n')
open('/kaggle/working/val.txt',   'w').write('\n'.join(val_p) + '\n')

DATA_YAML = '/kaggle/working/data_by_source.yaml'
with open(DATA_YAML, 'w') as f:
    f.write(f'path: {DEST}\n')
    f.write('train: /kaggle/working/train.txt\n')
    f.write('val: /kaggle/working/val.txt\n\n')
    f.write(f'nc: {len(names)}\nnames: {list(names)}\n')
print('\n' + open(DATA_YAML).read())

In [ ]:
# 5. Baseline BEFORE training -- the number the fine-tune has to beat.
#    COCO ids for the 5 classes that exist there; 'headphone' has no COCO
#    equivalent, so the baseline simply cannot detect it (mAP 0 by construction).
from ultralytics import YOLO

COCO_FOR = {'person': 0, 'tv': 62, 'laptop': 63, 'cell phone': 67, 'book': 73}
print('baseline can attempt:', [n for n in names if n in COCO_FOR])
print('baseline cannot see :', [n for n in names if n not in COCO_FOR])

baseline = YOLO('yolov8n.pt')  # 80-class COCO weights
print('\nBaseline is scored on the same val split after training, in cell 7.')

In [ ]:
# 6. Fine-tune.  ~2-3 h for 50 epochs on a T4; drop EPOCHS to smoke-test first.
EPOCHS = 50
IMGSZ  = 640
BATCH  = 32      # T4 handles 32 at 640 for yolov8n; lower it if you hit OOM

model = YOLO('yolov8n.pt')
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=2,
    project='/kaggle/working/runs',
    name='proctoring_yolov8n',
    exist_ok=True,
    patience=15,        # stop early if val stops improving -- val is a different
                        # person, so overfitting to the train subject shows up here
    seed=42,
)
print('best weights:', model.trainer.best)

In [ ]:
# 7. Per-class results on the held-out sources
best = YOLO(model.trainer.best)
metrics = best.val(data=DATA_YAML, split='val', device=0)

# IMPORTANT: box.ap50 / box.ap are indexed by the classes that actually APPEAR
# in val, not by the full class list. box.ap_class_index gives that mapping.
# Indexing them with enumerate(names) would misattribute every score after the
# first missing class -- likely here, since val is other people's footage and
# may contain no headphones or TVs at all.
present = {int(c): j for j, c in enumerate(metrics.box.ap_class_index)}

print(f"\n{'class':<14}{'mAP50':>9}{'mAP50-95':>11}")
print('-' * 34)
for i, name in enumerate(names):
    if i in present:
        j = present[i]
        print(f'{name:<14}{metrics.box.ap50[j]:>9.3f}{metrics.box.ap[j]:>11.3f}')
    else:
        print(f'{name:<14}{"absent from val":>20}')
print('-' * 34)
print(f"{'ALL':<14}{metrics.box.map50:>9.3f}{metrics.box.map:>11.3f}")

print('\nThe number that matters for Phase 1 is the cell phone row:')
print('pre-trained YOLOv8n managed F1 0.193 on this domain.')
print('A class marked "absent from val" was never tested -- no evidence either way.')

In [ ]:
# 8. Save the weights -- download from the notebook's Output tab when it finishes
import shutil
shutil.copy(model.trainer.best, '/kaggle/working/proctoring_yolov8n_best.pt')

# Drop the dataset so the output stays small enough to save
shutil.rmtree(DEST, ignore_errors=True)
print('ready to download: /kaggle/working/proctoring_yolov8n_best.pt')
print(f"{os.path.getsize('/kaggle/working/proctoring_yolov8n_best.pt')/1e6:.1f} MB")

## After it finishes

Download `proctoring_yolov8n_best.pt` into the repo at `models/detection/`, then locally:

```bash
# Re-calibrate the confidence threshold for the NEW model
python -m src.calibration.calibrate_phone_conf \
    --weights models/detection/proctoring_yolov8n_best.pt --finetuned \
    --export-root data/detection/external/roboflow_online_proctoring --split valid

# Then the real test: does it work on OUR webcam footage?
python -m src.detection.extract_frames --strategy missed \
    --weights models/detection/proctoring_yolov8n_best.pt --dry-run
```

That second command re-runs the measurement that started all of this. The pre-trained model
found a phone in **20.7%** of frames from our `phone_*` clips. If the fine-tune moves that
materially, Phase 2 worked.

### Honest limits of this run

- **Train is one person in one room.** ~87% of the dataset is a single video. The model will fit
  that subject well; the val score on other sources is the only evidence it generalises.
- **Val is ~3.3k images from 6 sources**, some tiny. A weak class result may mean the class is
  rare in val rather than that the model is bad — check the support counts.
- **`headphone` has no COCO equivalent**, so there's no baseline to compare it against. Any
  detection ability there is new capability rather than an improvement.
- **Train images are augmented ~3x** from ~7.5k unique frames, so effective diversity is lower
  than 21,867 suggests.